# Noise Removal And Parent-Conditioned Anomaly Labels

**Status:** research notebook, v1 experiment scaffold.  
**Source research note:** `notebooks/notes/anomaly.md`  
**Implementation plan:** `docs/plans/stage1_label_anomaly_research_plan_2026-05-20.md`

## Work Tracker

| Step | Status | Purpose | Main output |
|---|---|---|---|
| 1. Keep original target | Ready | `target_4class` remains source of truth | class map + collapse map |
| 2. Strong 4-class baseline | Ready | time-safe baseline before relabeling | collapsed-4 metrics + confusion matrix |
| 3. Out-of-fold probabilities | Ready | score each train row only when it is validation | OOF probability table |
| 4. K-means cluster diagnostics | Ready | test whether features organize into 4 class clusters and 2 direction clusters | cluster-quality charts |
| 5. Suspiciousness score | Ready | combine label conflict, disagreement, temporal inconsistency, and class-conditioned outlier score | per-row noise scores |
| 6. Class-local anomaly thresholding | Ready | top `2%..5%` within each original class | clean/anomaly/review action table |
| 7. Experimental 8-class target | Ready | map same-parent suspicious rows `k -> k+4` | `target_8class_anomaly` |
| 8. 8-class model + collapsed eval | Ready | test whether anomaly labels improve real objective | baseline vs anomaly comparison |

## Rules We Must Not Break

- `target_4class` is never overwritten.
- `4..7` are only same-parent anomaly leaves and must collapse back to `0..3` for primary evaluation.
- Time order is mandatory: train past -> validate future -> test later future.
- Out-of-fold scores must come from models that did not train on the scored row.
- Opposite-direction suspicious rows are not automatically converted to `k+4`; they are routed to review/exclude/low-weight handling.
- Keep the method only if collapsed-4 quality improves and cross-direction error falls or stays stable.

## Main Decision Criteria

Keep the anomaly setup only if the future holdout improves:

```text
collapsed_4_accuracy up
macro_f1_4 stable or up
direction_accuracy up
cross_direction_error down
```

Reject it if full 8-class accuracy improves while collapsed-4 quality or direction safety gets worse.

## Research Interpretation Used In This Notebook

The research note supports a **two-stage data-centric workflow**:

1. Use out-of-fold model probabilities and class-conditional geometry to find suspicious labels.
2. Relabel only conservative same-parent suspicious rows into anomaly leaves.

This notebook also adds your clustering idea:

- `k=4` clustering tests whether feature space naturally separates the four original classes.
- `k=2` clustering tests whether feature space at least separates direction: `DOWN={0,1}` vs `UP={2,3}`.
- K-means is diagnostic and contributes to anomaly scoring, but it does not decide labels alone.

The reason is practical: rare but valid market regimes can look like geometric outliers. We only mark anomalies when cluster evidence and out-of-fold prediction evidence agree.

In [1]:
from __future__ import annotations

import gc
import json
import math
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    confusion_matrix,
    f1_score,
    log_loss,
    normalized_mutual_info_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# Repository discovery
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# -----------------------------------------------------------------------------
# Experiment controls. Change only these for a different asset/root pilot.
# -----------------------------------------------------------------------------
TARGET_ASSET = "BTCUSDT"
CONTEXT_HASH = "corexself"
ROOT_ID = "8h_b"
TARGET_COL = "target_4class"

# Keep defaults bounded so the notebook is interactive. Increase for stronger runs.
USE_LAST_BATCHES = 900          # None = all selected merged batches.
MAX_ROWS = 180_000              # None = no row cap after batch selection.
MAX_MODEL_FEATURES = 250        # train-only top-variance feature limit.
OOF_FOLDS = 5
MIN_OOF_TRAIN_BATCHES = 120
ANOMALY_TOP_PCT = 0.03          # top 3% noisiest rows inside each original class.
OPPOSITE_HIGH_CONFIDENCE = 0.80
RANDOM_SEED = 42

# Faster notebook model settings. Stage-1 production CatBoost remains separate.
MODEL_BACKEND = "catboost"      # "catboost" with sklearn fallback if unavailable.
CATBOOST_ITERATIONS = 220
CATBOOST_DEPTH = 6
CATBOOST_LEARNING_RATE = 0.05
CATBOOST_THREAD_COUNT = 4          # avoids avoidable RAM spikes during notebook experiments.
CATBOOST_USED_RAM_LIMIT = None     # optional CatBoost soft limit; None avoids noisy warning spam.

CLASS_NAMES_4 = {
    0: "DOWN_BALANCED",
    1: "DOWN_EXPANSION",
    2: "UP_BALANCED",
    3: "UP_EXPANSION",
}
CLASS_NAMES_8 = {
    **CLASS_NAMES_4,
    4: "DOWN_BALANCED_ANOMALY",
    5: "DOWN_EXPANSION_ANOMALY",
    6: "UP_BALANCED_ANOMALY",
    7: "UP_EXPANSION_ANOMALY",
}
DIRECTION_NAMES = {0: "DOWN", 1: "UP"}

config_rows = [
    ("PROJECT_ROOT", str(PROJECT_ROOT)),
    ("TARGET_ASSET", TARGET_ASSET),
    ("CONTEXT_HASH", CONTEXT_HASH),
    ("ROOT_ID", ROOT_ID),
    ("USE_LAST_BATCHES", USE_LAST_BATCHES),
    ("MAX_ROWS", MAX_ROWS),
    ("MAX_MODEL_FEATURES", MAX_MODEL_FEATURES),
    ("OOF_FOLDS", OOF_FOLDS),
    ("ANOMALY_TOP_PCT", ANOMALY_TOP_PCT),
    ("MODEL_BACKEND", MODEL_BACKEND),
    ("CATBOOST_THREAD_COUNT", CATBOOST_THREAD_COUNT),
    ("CATBOOST_USED_RAM_LIMIT", CATBOOST_USED_RAM_LIMIT),
]
display(pd.DataFrame(config_rows, columns=["setting", "value"]))

,setting,value
0,PROJECT_ROOT,/run/media/przem/linux_data/risk_yield_multi-a...
1,TARGET_ASSET,BTCUSDT
2,CONTEXT_HASH,corexself
3,ROOT_ID,8h_b
4,USE_LAST_BATCHES,900
5,MAX_ROWS,180000
6,MAX_MODEL_FEATURES,250
7,OOF_FOLDS,5
8,ANOMALY_TOP_PCT,0.03
9,MODEL_BACKEND,catboost


In [2]:
# Locate generated merged Stage-1 datasets and select the configured target/root.
merged_root = PROJECT_ROOT / "data" / "htf_multiasset_merged"
manifest_paths = sorted(merged_root.glob("*/*/*/manifest.json"))
if not manifest_paths:
    raise FileNotFoundError(
        "No merged Stage-1 manifests found. Build one first with:\n"
        "python scripts/analysis/htf_stage1_regime_family_walkforward.py "
        "--build-merged-dataset --target-assets BTCUSDT "
        "--context-assets core-ex-target --roots 8h/B --plan-only"
    )

manifest_records = []
selected_manifest = None
for path in manifest_paths:
    manifest = json.loads(path.read_text())
    record = {
        "target_asset": manifest.get("target_asset"),
        "context_hash": manifest.get("context_hash"),
        "root_id": manifest.get("root_id"),
        "root": manifest.get("root"),
        "output_rows": manifest.get("output_rows"),
        "feature_columns_count": manifest.get("feature_columns_count"),
        "timestamp_min": manifest.get("timestamp_min"),
        "timestamp_max": manifest.get("timestamp_max"),
        "manifest_path": str(path),
    }
    manifest_records.append(record)
    if (
        record["target_asset"] == TARGET_ASSET
        and record["context_hash"] == CONTEXT_HASH
        and record["root_id"] == ROOT_ID
    ):
        selected_manifest = manifest

manifest_df = pd.DataFrame(manifest_records).sort_values(
    ["target_asset", "context_hash", "root_id"]
)
display(Markdown("### Available merged manifests"))
display(manifest_df)

if selected_manifest is None:
    raise FileNotFoundError(
        f"Configured merged dataset not found: target={TARGET_ASSET}, "
        f"context={CONTEXT_HASH}, root_id={ROOT_ID}. Available manifests are shown above."
    )

FEATURE_DIR = Path(selected_manifest["output_paths"]["features_dir"]) / "1m" / "target_4class"
LABEL_DIR = Path(selected_manifest["output_paths"]["labels_dir"]) / "1m"

display(Markdown("### Selected manifest"))
display(pd.DataFrame([selected_manifest]).T.rename(columns={0: "value"}).head(30))
print("Feature dir:", FEATURE_DIR)
print("Label dir  :", LABEL_DIR)

### Available merged manifests

,target_asset,context_hash,root_id,root,output_rows,feature_columns_count,timestamp_min,timestamp_max,manifest_path
0,BTCUSDT,3b6848353dc1,8h_b,8h/B,1342784,340,2021-03-29T00:00:00+00:00,2026-05-06T19:59:00+00:00,/run/media/przem/linux_data/risk_yield_multi-a...
1,BTCUSDT,corexself,8h_b,8h/B,413652,1162,2024-01-23T08:00:00+00:00,2026-05-06T19:59:00+00:00,/run/media/przem/linux_data/risk_yield_multi-a...


### Selected manifest

,value
generated_at,2026-05-20T02:36:59.229391+00:00
target_asset,BTCUSDT
context_assets,"[ETHUSDT, EURUSD, USDJPY, GC, CL, ES, NQ]"
context_hash,corexself
root,8h/B
root_id,8h_b
regime,8h
family,B
target_col,target_4class
timeframe,1m


Feature dir: /media/przem/linux_data/risk_yield_multi-asset_dataset/RiskYieldMM/data/htf_multiasset_merged/btcusdt/corexself/8h_b/features/1m/target_4class
Label dir  : /media/przem/linux_data/risk_yield_multi-asset_dataset/RiskYieldMM/data/htf_multiasset_merged/btcusdt/corexself/8h_b/labels/1m


In [3]:
# Discover merged feature/label batches and build lightweight metadata only.
#
# Memory note:
# This cell intentionally reads only timestamp/batch_id/label columns. Full
# feature columns are scanned one parquet batch at a time in the next cell to
# compute train-only variance, then only selected features are materialized.
feature_paths_all = sorted(FEATURE_DIR.glob("batch_*.parquet"))
if USE_LAST_BATCHES is not None:
    feature_paths = feature_paths_all[-int(USE_LAST_BATCHES):]
else:
    feature_paths = feature_paths_all
if not feature_paths:
    raise RuntimeError(f"No feature batches found under {FEATURE_DIR}")

feature_schema = pl.read_parquet(feature_paths[0], n_rows=0).schema
reserved_cols = {
    "timestamp",
    "batch_id",
    TARGET_COL,
    "target_name",
    "target_name_4",
    "direction_4",
    "direction_name",
    "row_id",
}
numeric_polars_dtypes = {
    pl.Float32,
    pl.Float64,
    pl.Int8,
    pl.Int16,
    pl.Int32,
    pl.Int64,
    pl.UInt8,
    pl.UInt16,
    pl.UInt32,
    pl.UInt64,
    pl.Boolean,
}
feature_cols_all = [
    col for col, dtype in feature_schema.items()
    if col not in reserved_cols and dtype in numeric_polars_dtypes
]

batch_meta_records: list[dict] = []
class_count_parts: list[pd.DataFrame] = []
missing_labels: list[str] = []
for feature_path in feature_paths:
    label_path = LABEL_DIR / feature_path.name
    if not label_path.exists():
        missing_labels.append(feature_path.name)
        continue
    f_meta = pl.read_parquet(feature_path, columns=["timestamp", "batch_id"])
    label_schema = pl.read_parquet(label_path, n_rows=0).schema
    label_cols = ["timestamp", "batch_id", TARGET_COL]
    if "target_name" in label_schema:
        label_cols.append("target_name")
    labels = pl.read_parquet(label_path, columns=label_cols)
    meta_joined = f_meta.join(labels, on=["timestamp", "batch_id"], how="inner").filter(
        pl.col(TARGET_COL).is_between(0, 3)
    )
    if meta_joined.is_empty():
        continue
    batch_id = int(meta_joined["batch_id"][0])
    batch_meta_records.append(
        {
            "batch_id": batch_id,
            "rows": meta_joined.height,
            "timestamp_min": meta_joined.select(pl.col("timestamp").min()).item(),
            "timestamp_max": meta_joined.select(pl.col("timestamp").max()).item(),
            "feature_path": str(feature_path),
            "label_path": str(label_path),
        }
    )
    class_count_parts.append(
        meta_joined.group_by(TARGET_COL).len().with_columns(pl.lit(batch_id).alias("batch_id")).to_pandas()
    )

if not batch_meta_records:
    raise RuntimeError("No valid feature/label rows were found.")

batch_meta = pd.DataFrame(batch_meta_records).sort_values("batch_id").reset_index(drop=True)
if MAX_ROWS is not None and batch_meta["rows"].sum() > MAX_ROWS:
    # Preserve the original notebook semantics: use the latest rows. Since each
    # batch is already chronological, selecting enough latest batches is an exact
    # superset and the final table applies `.tail(MAX_ROWS)` after loading.
    keep_ids = []
    running = 0
    for _, row in batch_meta.iloc[::-1].iterrows():
        keep_ids.append(int(row["batch_id"]))
        running += int(row["rows"])
        if running >= int(MAX_ROWS):
            break
    batch_meta = batch_meta[batch_meta["batch_id"].isin(keep_ids)].sort_values("batch_id").reset_index(drop=True)
    feature_paths = [Path(path) for path in batch_meta["feature_path"].to_list()]

class_counts = pd.concat(class_count_parts, ignore_index=True)
class_counts = class_counts[class_counts["batch_id"].isin(batch_meta["batch_id"])]
class_counts = class_counts.groupby(TARGET_COL, as_index=False)["len"].sum().sort_values(TARGET_COL)
direction_counts = pd.DataFrame(
    {
        "direction_name": ["DOWN", "UP"],
        "len": [
            int(class_counts[class_counts[TARGET_COL].isin([0, 1])]["len"].sum()),
            int(class_counts[class_counts[TARGET_COL].isin([2, 3])]["len"].sum()),
        ],
    }
)

summary = pd.DataFrame([
    ("loaded_rows_planned", int(batch_meta["rows"].sum() if MAX_ROWS is None else min(batch_meta["rows"].sum(), int(MAX_ROWS)))),
    ("loaded_batches_planned", int(batch_meta["batch_id"].nunique())),
    ("feature_columns_before_selection", len(feature_cols_all)),
    ("timestamp_min", batch_meta["timestamp_min"].min()),
    ("timestamp_max", batch_meta["timestamp_max"].max()),
    ("missing_label_batches", len(missing_labels)),
], columns=["metric", "value"])
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.barplot(data=class_counts, x=TARGET_COL, y="len", order=[0, 1, 2, 3], ax=axes[0])
axes[0].set_title("Original 4-class distribution")
axes[0].set_xticklabels([CLASS_NAMES_4[i].replace("_", "\n") for i in range(4)])

sns.barplot(data=direction_counts, x="direction_name", y="len", order=["DOWN", "UP"], ax=axes[1])
axes[1].set_title("Direction distribution")

axes[2].plot(batch_meta["batch_id"], batch_meta["rows"], linewidth=1)
axes[2].set_title("Rows per batch")
axes[2].set_xlabel("batch_id")
axes[2].set_ylabel("rows")
plt.tight_layout()
plt.show()

,metric,value
0,loaded_rows_planned,180000
1,loaded_batches_planned,766
2,feature_columns_before_selection,1162
3,timestamp_min,2025-05-08 00:00:00+00:00
4,timestamp_max,2026-05-06 19:59:00+00:00
5,missing_label_batches,0


In [4]:
# Metric and modeling utilities used by all later cells.
def collapse_8_to_4(values):
    arr = np.asarray(values, dtype=int)
    return np.where(arr >= 4, arr - 4, arr)


def to_direction(values):
    arr4 = collapse_8_to_4(values)
    return np.where(np.isin(arr4, [0, 1]), 0, 1)


def collapse_proba_8_to_4(proba8):
    proba8 = np.asarray(proba8, dtype=float)
    if proba8.shape[1] == 4:
        return proba8
    out = np.zeros((proba8.shape[0], 4), dtype=float)
    out[:, 0] = proba8[:, 0] + proba8[:, 4]
    out[:, 1] = proba8[:, 1] + proba8[:, 5]
    out[:, 2] = proba8[:, 2] + proba8[:, 6]
    out[:, 3] = proba8[:, 3] + proba8[:, 7]
    row_sum = out.sum(axis=1, keepdims=True)
    return np.divide(out, row_sum, out=np.full_like(out, 0.25), where=row_sum > 0)


def compute_eval_metrics(y_true4, pred_labels, pred_proba=None, prefix=""):
    y_true4 = np.asarray(y_true4, dtype=int)
    pred4 = collapse_8_to_4(pred_labels)
    true_dir = to_direction(y_true4)
    pred_dir = to_direction(pred4)
    metrics = {
        f"{prefix}collapsed_4_accuracy": accuracy_score(y_true4, pred4),
        f"{prefix}macro_f1_4": f1_score(y_true4, pred4, labels=[0, 1, 2, 3], average="macro", zero_division=0),
        f"{prefix}direction_accuracy": accuracy_score(true_dir, pred_dir),
        f"{prefix}cross_direction_error": float(np.mean(true_dir != pred_dir)),
    }
    if pred_proba is not None:
        proba4 = collapse_proba_8_to_4(pred_proba)
        metrics[f"{prefix}logloss_4"] = log_loss(y_true4, proba4, labels=[0, 1, 2, 3])
    return metrics


def plot_confusion(y_true4, pred_labels, title):
    pred4 = collapse_8_to_4(pred_labels)
    cm = confusion_matrix(y_true4, pred4, labels=[0, 1, 2, 3])
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm,
        annot=cm,
        fmt="d",
        cmap="Blues",
        xticklabels=[CLASS_NAMES_4[i].replace("_", "\n") for i in range(4)],
        yticklabels=[CLASS_NAMES_4[i].replace("_", "\n") for i in range(4)],
    )
    plt.title(title)
    plt.xlabel("Predicted collapsed class")
    plt.ylabel("True class")
    plt.tight_layout()
    plt.show()


def train_model(X_df, y, *, n_classes, sample_weight=None, seed=RANDOM_SEED):
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(X_df).astype(np.float32, copy=False)
    backend = MODEL_BACKEND
    try:
        if backend != "catboost":
            raise ImportError("CatBoost disabled by MODEL_BACKEND")
        from catboost import CatBoostClassifier
        catboost_params = {
            "loss_function": "MultiClass",
            "classes_count": int(n_classes),
            "iterations": CATBOOST_ITERATIONS,
            "depth": CATBOOST_DEPTH,
            "learning_rate": CATBOOST_LEARNING_RATE,
            "random_seed": int(seed),
            "verbose": False,
            "allow_writing_files": False,
            "task_type": "CPU",
            "thread_count": CATBOOST_THREAD_COUNT,
        }
        if CATBOOST_USED_RAM_LIMIT:
            catboost_params["used_ram_limit"] = CATBOOST_USED_RAM_LIMIT
        model = CatBoostClassifier(**catboost_params)
        model.fit(X, y, sample_weight=sample_weight)
        backend_used = "catboost"
    except Exception as exc:
        from sklearn.ensemble import HistGradientBoostingClassifier
        print(f"CatBoost unavailable or failed ({exc}); using HistGradientBoostingClassifier fallback.")
        model = HistGradientBoostingClassifier(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=int(seed),
        )
        model.fit(X, y, sample_weight=sample_weight)
        backend_used = "hist_gradient_boosting"
    return {"model": model, "imputer": imputer, "n_classes": int(n_classes), "backend": backend_used}


def predict_proba_model(bundle, X_df):
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    X = bundle["imputer"].transform(X_df).astype(np.float32, copy=False)
    raw = np.asarray(bundle["model"].predict_proba(X), dtype=float)
    n_classes = int(bundle["n_classes"])
    full = np.zeros((len(X_df), n_classes), dtype=float)
    classes = getattr(bundle["model"], "classes_", np.arange(raw.shape[1]))
    for raw_idx, cls in enumerate(np.asarray(classes, dtype=int)):
        if 0 <= cls < n_classes:
            full[:, cls] = raw[:, raw_idx]
    row_sum = full.sum(axis=1, keepdims=True)
    return np.divide(full, row_sum, out=np.full_like(full, 1.0 / n_classes), where=row_sum > 0)

# Cost matrix used for analysis and future cost-sensitive objectives.
cost = np.zeros((8, 8), dtype=float)
for i in range(8):
    for j in range(8):
        if i == j:
            cost[i, j] = 0.0
        elif collapse_8_to_4([i])[0] == collapse_8_to_4([j])[0]:
            cost[i, j] = 0.35
        elif to_direction([i])[0] == to_direction([j])[0]:
            cost[i, j] = 1.0
        else:
            cost[i, j] = 4.0

plt.figure(figsize=(8, 6))
sns.heatmap(cost, annot=True, fmt=".2g", cmap="Reds", cbar=False)
plt.title("Mistake cost matrix: opposite direction is expensive")
plt.xlabel("Predicted 8-class label")
plt.ylabel("True 8-class label")
plt.tight_layout()
plt.show()

display(pd.DataFrame({"class_id": list(CLASS_NAMES_8), "name": list(CLASS_NAMES_8.values())}))

,class_id,name
0,0,DOWN_BALANCED
1,1,DOWN_EXPANSION
2,2,UP_BALANCED
3,3,UP_EXPANSION
4,4,DOWN_BALANCED_ANOMALY
5,5,DOWN_EXPANSION_ANOMALY
6,6,UP_BALANCED_ANOMALY
7,7,UP_EXPANSION_ANOMALY


## Step 1-2: Time-Safe Split And 4-Class Baseline

This split is chronological by `batch_id`:

```text
train past -> validation future -> test later future
```

Feature selection is also fit on the training period only. This cell uses top training variance as a conservative first selector for notebook speed; production Stage-1 feature policy remains separate.

In [5]:
# Chronological split by observed batch ids.
unique_batches = np.array(sorted(batch_meta["batch_id"].unique()))
if len(unique_batches) < 10:
    raise RuntimeError("Not enough batches for chronological train/val/test split.")

train_cut = int(len(unique_batches) * 0.60)
val_cut = int(len(unique_batches) * 0.80)
train_batches = set(unique_batches[:train_cut])
val_batches = set(unique_batches[train_cut:val_cut])
test_batches = set(unique_batches[val_cut:])

batch_meta["split"] = np.select(
    [batch_meta["batch_id"].isin(train_batches), batch_meta["batch_id"].isin(val_batches), batch_meta["batch_id"].isin(test_batches)],
    ["train", "val", "test"],
    default="unused",
)

# Train-only feature variance selector, computed one parquet batch at a time.
# This preserves the same variance-selection rule without holding all 1,162
# features in memory at once.
n_features_all = len(feature_cols_all)
feature_sum = np.zeros(n_features_all, dtype=np.float64)
feature_sumsq = np.zeros(n_features_all, dtype=np.float64)
feature_count = np.zeros(n_features_all, dtype=np.float64)
train_feature_paths = batch_meta.loc[batch_meta["split"] == "train", "feature_path"].to_list()
for path in train_feature_paths:
    batch = pl.read_parquet(path, columns=feature_cols_all)
    arr = batch.to_numpy().astype(np.float64, copy=False)
    finite = np.isfinite(arr)
    arr_clean = np.where(finite, arr, 0.0)
    feature_sum += arr_clean.sum(axis=0)
    feature_sumsq += (arr_clean * arr_clean).sum(axis=0)
    feature_count += finite.sum(axis=0)
    del batch, arr, finite, arr_clean

valid = feature_count > 1
variance_values = np.full(n_features_all, np.nan, dtype=np.float64)
variance_values[valid] = (feature_sumsq[valid] - (feature_sum[valid] ** 2 / feature_count[valid])) / (feature_count[valid] - 1.0)
variance = (
    pd.Series(variance_values, index=feature_cols_all, dtype="float64")
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sort_values(ascending=False)
)
selected_features = variance.head(min(MAX_MODEL_FEATURES, len(variance))).index.to_list()
feature_cols = selected_features

del feature_sum, feature_sumsq, feature_count, variance_values
gc.collect()

# Load only selected features and labels. This is the first full row-level table
# materialized in Pandas, and it is intentionally selected-width.
frames: list[pl.DataFrame] = []
for _, row in batch_meta.iterrows():
    feature_path = Path(row["feature_path"])
    label_path = Path(row["label_path"])
    features = pl.read_parquet(feature_path, columns=["timestamp", "batch_id", *selected_features])
    labels = pl.read_parquet(label_path, columns=["timestamp", "batch_id", TARGET_COL])
    frames.append(features.join(labels, on=["timestamp", "batch_id"], how="inner"))

selected_df = pl.concat(frames, how="vertical_relaxed").sort(["batch_id", "timestamp"])
del frames
gc.collect()

selected_df = selected_df.filter(pl.col(TARGET_COL).is_between(0, 3))
if MAX_ROWS is not None and selected_df.height > MAX_ROWS:
    selected_df = selected_df.tail(int(MAX_ROWS))

selected_df = selected_df.with_row_index("row_id").with_columns(
    [
        pl.col(TARGET_COL).replace_strict(CLASS_NAMES_4).alias("target_name_4"),
        pl.when(pl.col(TARGET_COL).is_in([0, 1])).then(pl.lit(0)).otherwise(pl.lit(1)).alias("direction_4"),
    ]
).with_columns(
    pl.col("direction_4").replace_strict(DIRECTION_NAMES).alias("direction_name")
)

pdf = selected_df.to_pandas()
del selected_df
gc.collect()

feature_block = pdf[selected_features].apply(pd.to_numeric, errors="coerce", downcast="float")
pdf = pd.concat([pdf.drop(columns=selected_features), feature_block], axis=1).copy()

pdf["split"] = np.select(
    [pdf["batch_id"].isin(train_batches), pdf["batch_id"].isin(val_batches), pdf["batch_id"].isin(test_batches)],
    ["train", "val", "test"],
    default="unused",
)
train_df = pdf.loc[pdf["split"] == "train"].reset_index(drop=True)
val_df = pdf.loc[pdf["split"] == "val"].reset_index(drop=True)
test_df = pdf.loc[pdf["split"] == "test"].reset_index(drop=True)

split_summary = pdf.groupby("split").agg(
    rows=("row_id", "count"),
    batches=("batch_id", "nunique"),
    ts_min=("timestamp", "min"),
    ts_max=("timestamp", "max"),
).reset_index()
display(split_summary)

top_var = variance.head(30).reset_index()
top_var.columns = ["feature", "train_variance"]
display(Markdown(f"### Selected {len(selected_features)} model features from train-only variance"))
display(top_var)

memory_summary = pd.DataFrame([
    ("pandas_rows", len(pdf)),
    ("selected_feature_columns", len(selected_features)),
    ("pandas_memory_mb", round(pdf.memory_usage(deep=True).sum() / 1024**2, 2)),
], columns=["metric", "value"])
display(Markdown("### Memory-safe selected-width table"))
display(memory_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.countplot(data=pdf, x="split", order=["train", "val", "test"], ax=axes[0])
axes[0].set_title("Rows by chronological split")

class_by_split = pd.crosstab(pdf["split"], pdf[TARGET_COL], normalize="index").reindex(["train", "val", "test"])
class_by_split.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab10")
axes[1].set_title("Class mix by split")
axes[1].set_ylabel("share")
axes[1].legend(title="target_4class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

,split,rows,batches,ts_min,ts_max
0,test,36595,154,2026-02-23 16:00:00+00:00,2026-05-06 19:59:00+00:00
1,train,107553,459,2025-05-08 01:15:00+00:00,2025-12-10 03:59:00+00:00
2,val,35852,153,2025-12-10 08:00:00+00:00,2026-02-23 11:59:00+00:00


### Selected 250 model features from train-only variance

,feature,train_variance
0,C_EURUSD__H_4class_1_ou_zscore,2.583631e+06
1,C_EURUSD__H_4class_1_ou_zscore_abs,2.583622e+06
2,T_BTCUSDT__H_4class_1_ou_kappa,1.350110e+01
3,C_ETHUSDT__H_4class_1_ou_kappa,1.306664e+01
4,C_GC__H_4class_1_ou_kappa,1.278126e+01
5,C_NQ__H_4class_1_ou_kappa,1.272713e+01
6,C_ES__H_4class_1_ou_kappa,1.244032e+01
7,C_CL__H_4class_1_ou_kappa,1.227989e+01
8,C_EURUSD__H_4class_1_ou_kappa,1.182115e+01
9,C_USDJPY__H_4class_1_ou_kappa,1.042565e+01


### Memory-safe selected-width table

,metric,value
0,pandas_rows,180000.00
1,selected_feature_columns,250.00
2,pandas_memory_mb,209.91


In [6]:
# Train the normal 4-class baseline and evaluate on untouched future splits.
baseline = train_model(
    train_df[selected_features],
    train_df[TARGET_COL].astype(int).to_numpy(),
    n_classes=4,
)

baseline_results = []
baseline_predictions = {}
for split_name, split_df in [("val", val_df), ("test", test_df)]:
    proba = predict_proba_model(baseline, split_df[selected_features])
    pred = proba.argmax(axis=1)
    metrics = compute_eval_metrics(
        split_df[TARGET_COL].astype(int).to_numpy(),
        pred,
        proba,
        prefix=f"{split_name}_",
    )
    baseline_results.append({"model": "baseline_4class", "split": split_name, **metrics})
    baseline_predictions[split_name] = {"proba": proba, "pred": pred}

baseline_metrics_df = pd.DataFrame(baseline_results)
display(Markdown(f"### Baseline backend: `{baseline['backend']}`"))
display(baseline_metrics_df.T)

metric_cols = [c for c in baseline_metrics_df.columns if c not in {"model", "split"}]
plot_df = baseline_metrics_df.melt(id_vars=["model", "split"], value_vars=metric_cols, var_name="metric", value_name="value")
plt.figure(figsize=(14, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="split")
plt.xticks(rotation=35, ha="right")
plt.title("4-class baseline metrics")
plt.tight_layout()
plt.show()

plot_confusion(
    val_df[TARGET_COL].astype(int).to_numpy(),
    baseline_predictions["val"]["pred"],
    "Baseline validation confusion matrix",
)
plot_confusion(
    test_df[TARGET_COL].astype(int).to_numpy(),
    baseline_predictions["test"]["pred"],
    "Baseline test confusion matrix",
)

### Baseline backend: `catboost`

,0,1
model,baseline_4class,baseline_4class
split,val,test
val_collapsed_4_accuracy,0.319173,NaN
val_macro_f1_4,0.242538,NaN
val_direction_accuracy,0.544712,NaN
val_cross_direction_error,0.455288,NaN
val_logloss_4,1.404008,NaN
test_collapsed_4_accuracy,NaN,0.314633
test_macro_f1_4,NaN,0.21887
test_direction_accuracy,NaN,0.515344


## Step 3: Out-Of-Fold Probabilities For Training Rows

Noise scoring must use predictions from models that did **not** train on the scored row. This cell creates expanding walk-forward OOF folds inside the training period.

Rows before the first warmup window cannot receive an OOF score in this simple notebook version and are kept clean by default.

In [7]:
def make_expanding_oof_folds(batch_ids, *, n_folds, min_train_batches):
    batches = np.array(sorted(batch_ids))
    if len(batches) <= min_train_batches + n_folds:
        min_train_batches = max(3, len(batches) // 3)
    val_span = max(1, (len(batches) - min_train_batches) // max(n_folds, 1))
    folds = []
    start = min_train_batches
    fold_id = 0
    while start < len(batches) and fold_id < n_folds:
        end = min(len(batches), start + val_span)
        train_part = batches[:start]
        val_part = batches[start:end]
        if len(train_part) and len(val_part):
            folds.append((fold_id, train_part, val_part))
        start = end
        fold_id += 1
    return folds

oof_folds = make_expanding_oof_folds(
    sorted(train_batches),
    n_folds=OOF_FOLDS,
    min_train_batches=MIN_OOF_TRAIN_BATCHES,
)

fold_rows = []
oof_parts = []
for fold_id, fold_train_batches, fold_val_batches in oof_folds:
    fold_train = train_df[train_df["batch_id"].isin(fold_train_batches)].copy()
    fold_val = train_df[train_df["batch_id"].isin(fold_val_batches)].copy()
    if fold_train.empty or fold_val.empty:
        continue
    fold_model = train_model(
        fold_train[selected_features],
        fold_train[TARGET_COL].astype(int).to_numpy(),
        n_classes=4,
        seed=RANDOM_SEED + fold_id + 1,
    )
    proba = predict_proba_model(fold_model, fold_val[selected_features])
    pred = proba.argmax(axis=1)
    part = fold_val[["row_id", "timestamp", "batch_id", TARGET_COL, "direction_4"]].copy()
    for cls in range(4):
        part[f"prob_{cls}"] = proba[:, cls].astype(np.float32)
    part["pred_label"] = pred.astype(np.int8)
    part["model_confidence"] = proba.max(axis=1).astype(np.float32)
    part["fold_id"] = fold_id
    part["oof_backend"] = fold_model["backend"]
    oof_parts.append(part)
    fold_metrics = compute_eval_metrics(part[TARGET_COL].to_numpy(), pred, proba, prefix="oof_")
    fold_rows.append({
        "fold_id": fold_id,
        "train_batches": len(fold_train_batches),
        "val_batches": len(fold_val_batches),
        "val_rows": len(fold_val),
        **fold_metrics,
    })
    del fold_train, fold_val, fold_model, proba, pred
    gc.collect()

oof_df = pd.concat(oof_parts, ignore_index=True) if oof_parts else pd.DataFrame()
fold_summary = pd.DataFrame(fold_rows)
display(Markdown("### OOF fold summary"))
display(fold_summary)

coverage = pd.DataFrame([
    ("train_rows", len(train_df)),
    ("oof_scored_rows", len(oof_df)),
    ("coverage_pct", 100 * len(oof_df) / max(len(train_df), 1)),
], columns=["metric", "value"])
display(coverage)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
if not fold_summary.empty:
    sns.lineplot(data=fold_summary, x="fold_id", y="oof_collapsed_4_accuracy", marker="o", ax=axes[0], label="accuracy")
    sns.lineplot(data=fold_summary, x="fold_id", y="oof_cross_direction_error", marker="o", ax=axes[0], label="cross_dir_error")
axes[0].set_title("OOF fold metrics")
axes[0].legend()

if not oof_df.empty:
    sns.histplot(oof_df["model_confidence"], bins=40, ax=axes[1])
axes[1].set_title("OOF model confidence")

if not oof_df.empty:
    sns.countplot(data=oof_df, x=TARGET_COL, order=[0, 1, 2, 3], ax=axes[2])
axes[2].set_title("OOF scored rows by class")
plt.tight_layout()
plt.show()

### OOF fold summary

,fold_id,train_batches,val_batches,val_rows,oof_collapsed_4_accuracy,oof_macro_f1_4,oof_direction_accuracy,oof_cross_direction_error,oof_logloss_4
0,0,120,67,15653,0.368364,0.266830,0.529994,0.470006,1.375020
1,1,187,67,15779,0.400469,0.250551,0.560935,0.439065,1.379438
2,2,254,67,15576,0.414291,0.233195,0.489407,0.510593,1.273388
3,3,321,67,15995,0.359487,0.231695,0.567490,0.432510,1.450040
4,4,388,67,15690,0.312365,0.216949,0.516699,0.483301,1.468085


,metric,value
0,train_rows,107553.000000
1,oof_scored_rows,78693.000000
2,coverage_pct,73.166718


## Step 4: K-Means Cluster Diagnostics

This section tests your clustering idea directly.

For multiple feature counts, we fit train-only K-means and check future validation alignment:

- `k=4`: does feature space separate the original four classes?
- `k=2`: does feature space separate direction?

The goal is not to force K-means to become the classifier. The goal is to find feature sets where labels are geometrically coherent enough to support anomaly detection.

In [8]:
def map_clusters_to_labels(cluster_ids, labels):
    mapping = {}
    for cluster in np.unique(cluster_ids):
        mask = cluster_ids == cluster
        if not np.any(mask):
            mapping[int(cluster)] = -1
            continue
        values, counts = np.unique(np.asarray(labels)[mask], return_counts=True)
        mapping[int(cluster)] = int(values[np.argmax(counts)])
    return mapping


def cluster_eval_for_features(feature_subset, *, k, label_col, train_sample, val_sample):
    scaler = StandardScaler()
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(train_sample[feature_subset].replace([np.inf, -np.inf], np.nan)).astype(np.float32, copy=False)
    X_val = imputer.transform(val_sample[feature_subset].replace([np.inf, -np.inf], np.nan)).astype(np.float32, copy=False)
    X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)
    X_val = scaler.transform(X_val).astype(np.float32, copy=False)
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_SEED, batch_size=4096, n_init="auto")
    train_cluster = km.fit_predict(X_train)
    val_cluster = km.predict(X_val)
    mapping = map_clusters_to_labels(train_cluster, train_sample[label_col].to_numpy())
    val_mapped = np.array([mapping.get(int(c), -1) for c in val_cluster], dtype=int)
    return {
        "k": k,
        "feature_count": len(feature_subset),
        "train_ari": adjusted_rand_score(train_sample[label_col], train_cluster),
        "val_ari": adjusted_rand_score(val_sample[label_col], val_cluster),
        "train_nmi": normalized_mutual_info_score(train_sample[label_col], train_cluster),
        "val_nmi": normalized_mutual_info_score(val_sample[label_col], val_cluster),
        "val_cluster_mapped_accuracy": accuracy_score(val_sample[label_col], val_mapped),
    }

cluster_train = (train_df.sample(min(len(train_df), 60_000), random_state=RANDOM_SEED) if len(train_df) > 60_000 else train_df).copy()
cluster_val = (val_df.sample(min(len(val_df), 40_000), random_state=RANDOM_SEED) if len(val_df) > 40_000 else val_df).copy()
cluster_train["direction_tmp"] = to_direction(cluster_train[TARGET_COL].to_numpy())
cluster_val["direction_tmp"] = to_direction(cluster_val[TARGET_COL].to_numpy())

feature_counts = sorted(set([25, 50, 100, 150, min(250, len(selected_features))]))
cluster_rows = []
for n_features in feature_counts:
    subset = selected_features[:min(n_features, len(selected_features))]
    cluster_rows.append(cluster_eval_for_features(subset, k=4, label_col=TARGET_COL, train_sample=cluster_train, val_sample=cluster_val))
    cluster_rows.append(cluster_eval_for_features(subset, k=2, label_col="direction_tmp", train_sample=cluster_train, val_sample=cluster_val))

cluster_metrics_df = pd.DataFrame(cluster_rows)
display(cluster_metrics_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.lineplot(data=cluster_metrics_df, x="feature_count", y="val_nmi", hue="k", marker="o", ax=axes[0])
axes[0].set_title("Validation normalized mutual information")
sns.lineplot(data=cluster_metrics_df, x="feature_count", y="val_ari", hue="k", marker="o", ax=axes[1])
axes[1].set_title("Validation adjusted rand index")
sns.lineplot(data=cluster_metrics_df, x="feature_count", y="val_cluster_mapped_accuracy", hue="k", marker="o", ax=axes[2])
axes[2].set_title("Validation cluster->label mapped accuracy")
plt.tight_layout()
plt.show()

best_k4 = cluster_metrics_df[cluster_metrics_df["k"] == 4].sort_values("val_nmi", ascending=False).head(1)
best_k2 = cluster_metrics_df[cluster_metrics_df["k"] == 2].sort_values("val_nmi", ascending=False).head(1)
display(Markdown("### Best clustering feature counts"))
display(pd.concat([best_k4.assign(view="4-class"), best_k2.assign(view="2-direction")]))

,k,feature_count,train_ari,val_ari,train_nmi,val_nmi,val_cluster_mapped_accuracy
0,4,25,0.000532,0.003501,0.000794,0.001575,0.263472
1,2,25,0.000006,0.000689,0.000002,0.000170,0.524239
2,4,50,0.001238,0.002578,0.000264,0.001210,0.272091
3,2,50,0.000039,-0.000020,0.000020,0.000051,0.524239
4,4,100,0.000685,0.001058,0.000128,0.000460,0.278171
5,2,100,-0.000032,0.000095,0.000002,0.000013,0.524239
6,4,150,0.005204,0.003029,0.004427,0.004813,0.311838
7,2,150,0.000268,0.004554,0.000250,0.004132,0.524239
8,4,250,0.005878,0.005327,0.005433,0.006425,0.311503
9,2,250,0.002804,0.006024,0.001794,0.003816,0.539022


### Best clustering feature counts

,k,feature_count,train_ari,val_ari,train_nmi,val_nmi,val_cluster_mapped_accuracy,view
8,4,250,0.005878,0.005327,0.005433,0.006425,0.311503,4-class
7,2,150,0.000268,0.004554,0.000250,0.004132,0.524239,2-direction


In [9]:
# Visualize validation rows in a 2D PCA projection with the best k=4 clustering setup.
if cluster_metrics_df.empty:
    raise RuntimeError("Run the clustering metrics cell first.")

best_feature_count = int(cluster_metrics_df[cluster_metrics_df["k"] == 4].sort_values("val_nmi", ascending=False).iloc[0]["feature_count"])
cluster_features = selected_features[:best_feature_count]

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train = imputer.fit_transform(cluster_train[cluster_features].replace([np.inf, -np.inf], np.nan)).astype(np.float32, copy=False)
X_val = imputer.transform(cluster_val[cluster_features].replace([np.inf, -np.inf], np.nan)).astype(np.float32, copy=False)
X_train_s = scaler.fit_transform(X_train).astype(np.float32, copy=False)
X_val_s = scaler.transform(X_val).astype(np.float32, copy=False)

km4 = MiniBatchKMeans(n_clusters=4, random_state=RANDOM_SEED, batch_size=4096, n_init="auto")
train_cluster4 = km4.fit_predict(X_train_s)
val_cluster4 = km4.predict(X_val_s)
cluster_map4 = map_clusters_to_labels(train_cluster4, cluster_train[TARGET_COL].to_numpy())
val_cluster_mapped4 = np.array([cluster_map4.get(int(c), -1) for c in val_cluster4])

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pca.fit(X_train_s)
val_xy = pca.transform(X_val_s)
plot_sample = pd.DataFrame({
    "pc1": val_xy[:, 0],
    "pc2": val_xy[:, 1],
    "true_class": cluster_val[TARGET_COL].map(CLASS_NAMES_4).to_numpy(),
    "cluster_id": val_cluster4,
    "cluster_mapped_class": [CLASS_NAMES_4.get(int(x), "UNKNOWN") for x in val_cluster_mapped4],
})
if len(plot_sample) > 15_000:
    plot_sample = plot_sample.sample(15_000, random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=plot_sample, x="pc1", y="pc2", hue="true_class", s=8, alpha=0.45, ax=axes[0], linewidth=0)
axes[0].set_title(f"Validation PCA by true class, top {best_feature_count} features")
axes[0].legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")

sns.scatterplot(data=plot_sample, x="pc1", y="pc2", hue="cluster_mapped_class", s=8, alpha=0.45, ax=axes[1], linewidth=0)
axes[1].set_title("Validation PCA by mapped k=4 cluster")
axes[1].legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display(pd.DataFrame([{"cluster_id": k, "mapped_to_class": CLASS_NAMES_4.get(v, "UNKNOWN")} for k, v in cluster_map4.items()]))

del X_train, X_val, X_train_s, X_val_s, val_xy
gc.collect()

,cluster_id,mapped_to_class
0,0,DOWN_BALANCED
1,1,UP_BALANCED
2,2,DOWN_BALANCED
3,3,UP_BALANCED


1584

## Step 5-6: Suspiciousness Score And Class-Local Anomaly Decisions

First implementation score:

```text
noise_score =
  0.50 * label_conflict_score
+ 0.20 * ensemble_disagreement_proxy
+ 0.15 * temporal_inconsistency
+ 0.15 * class_conditional_outlier_score
```

Notes:

- `label_conflict_score = 1 - P(true_class)` from OOF probabilities.
- `ensemble_disagreement_proxy` is `1` when OOF prediction disagrees with the label. Later we can replace this with real multi-model disagreement.
- `class_conditional_outlier_score` comes from train-only per-class K-means distance percentile.
- Thresholding is class-local: top `ANOMALY_TOP_PCT` inside each original class.

In [10]:
def centered_direction_inconsistency(frame, window=9):
    ordered = frame.sort_values(["batch_id", "timestamp"]).copy()
    direction = ordered["direction_4"].astype(float)
    rolling_sum = direction.rolling(window=window, center=True, min_periods=2).sum() - direction
    rolling_count = direction.rolling(window=window, center=True, min_periods=2).count() - 1
    neighbor_up_rate = (rolling_sum / rolling_count.replace(0, np.nan)).fillna(direction)
    inconsistency = np.where(direction == 1.0, 1.0 - neighbor_up_rate, neighbor_up_rate)
    ordered["temporal_inconsistency"] = np.clip(inconsistency, 0.0, 1.0)
    return ordered[["row_id", "temporal_inconsistency"]]


def class_conditional_outlier_scores(train_source, score_source, feature_subset):
    rows = []
    for cls in [0, 1, 2, 3]:
        cls_train = train_source[train_source[TARGET_COL] == cls]
        cls_score = score_source[score_source[TARGET_COL] == cls]
        if cls_train.empty or cls_score.empty:
            continue
        n_clusters = min(4, max(1, len(cls_train) // 500))
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        X_train = imputer.fit_transform(cls_train[feature_subset].replace([np.inf, -np.inf], np.nan)).astype(np.float32, copy=False)
        X_score = imputer.transform(cls_score[feature_subset].replace([np.inf, -np.inf], np.nan)).astype(np.float32, copy=False)
        X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)
        X_score = scaler.transform(X_score).astype(np.float32, copy=False)
        km = MiniBatchKMeans(n_clusters=n_clusters, random_state=RANDOM_SEED + cls, batch_size=4096, n_init="auto")
        km.fit(X_train)
        train_dist = km.transform(X_train).min(axis=1)
        score_dist = km.transform(X_score).min(axis=1)
        # Percentile against same-class train distances.
        pct = np.searchsorted(np.sort(train_dist), score_dist, side="right") / max(len(train_dist), 1)
        rows.append(pd.DataFrame({"row_id": cls_score["row_id"].to_numpy(), "class_conditional_outlier_score": pct.astype(np.float32)}))
        del cls_train, cls_score, X_train, X_score, train_dist, score_dist, km
        gc.collect()
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["row_id", "class_conditional_outlier_score"])

if oof_df.empty:
    raise RuntimeError("OOF predictions are required before scoring label noise.")

score_df = oof_df.copy()
for cls in range(4):
    score_df.loc[:, f"prob_{cls}"] = score_df[f"prob_{cls}"].astype(np.float32)
prob_matrix = score_df[[f"prob_{cls}" for cls in range(4)]].to_numpy(dtype=np.float32, copy=False)
true_idx = score_df[TARGET_COL].to_numpy(dtype=np.int64, copy=False)
score_df["p_true"] = prob_matrix[np.arange(len(score_df)), true_idx].astype(np.float32)
del prob_matrix, true_idx
score_df["label_conflict_score"] = 1.0 - score_df["p_true"]
score_df["ensemble_disagreement_proxy"] = (score_df["pred_label"] != score_df[TARGET_COL]).astype(float)

score_df = score_df.merge(centered_direction_inconsistency(train_df), on="row_id", how="left")
score_source = train_df.loc[train_df["row_id"].isin(score_df["row_id"]), ["row_id", TARGET_COL, *selected_features[:best_feature_count]]].copy()
outlier_df = class_conditional_outlier_scores(train_df, score_source, selected_features[:best_feature_count])
score_df = score_df.merge(outlier_df, on="row_id", how="left")
del score_source, outlier_df
gc.collect()
score_df["temporal_inconsistency"] = score_df["temporal_inconsistency"].fillna(0.0)
score_df["class_conditional_outlier_score"] = score_df["class_conditional_outlier_score"].fillna(0.0)

# Rank-normalize signals within each original class so naturally noisy classes do not dominate globally.
signal_cols = [
    "label_conflict_score",
    "ensemble_disagreement_proxy",
    "temporal_inconsistency",
    "class_conditional_outlier_score",
]
for col in signal_cols:
    score_df[f"{col}_rank"] = score_df.groupby(TARGET_COL)[col].rank(pct=True, method="average")

score_df["noise_score"] = (
    0.50 * score_df["label_conflict_score_rank"]
    + 0.20 * score_df["ensemble_disagreement_proxy_rank"]
    + 0.15 * score_df["temporal_inconsistency_rank"]
    + 0.15 * score_df["class_conditional_outlier_score_rank"]
)
score_df["true_direction"] = to_direction(score_df[TARGET_COL].to_numpy())
score_df["pred_direction"] = to_direction(score_df["pred_label"].to_numpy())
score_df["opposite_direction_prediction"] = score_df["true_direction"] != score_df["pred_direction"]

thresholds = (
    score_df.groupby(TARGET_COL)["noise_score"]
    .quantile(1.0 - ANOMALY_TOP_PCT)
    .rename("class_threshold")
    .reset_index()
)
score_df = score_df.merge(thresholds, on=TARGET_COL, how="left")
score_df["suspicious"] = score_df["noise_score"] >= score_df["class_threshold"]

score_summary = score_df.groupby(TARGET_COL).agg(
    rows=("row_id", "count"),
    p_true_mean=("p_true", "mean"),
    noise_mean=("noise_score", "mean"),
    threshold=("class_threshold", "first"),
    suspicious_rows=("suspicious", "sum"),
    opposite_direction_rate=("opposite_direction_prediction", "mean"),
).reset_index()
score_summary["class_name"] = score_summary[TARGET_COL].map(CLASS_NAMES_4)
display(score_summary)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
sns.histplot(data=score_df, x="p_true", hue=TARGET_COL, bins=40, element="step", stat="density", common_norm=False, ax=axes[0, 0])
axes[0, 0].set_title("OOF probability assigned to observed label")
sns.histplot(data=score_df, x="noise_score", hue=TARGET_COL, bins=40, element="step", stat="density", common_norm=False, ax=axes[0, 1])
axes[0, 1].set_title("Class-rank normalized noise score")
sns.scatterplot(data=score_df.sample(min(len(score_df), 12_000), random_state=RANDOM_SEED), x="p_true", y="noise_score", hue="suspicious", alpha=0.4, s=12, ax=axes[1, 0])
axes[1, 0].set_title("Low p_true + high score candidates")
sns.barplot(data=score_summary, x="class_name", y="suspicious_rows", ax=axes[1, 1])
axes[1, 1].set_title(f"Top {ANOMALY_TOP_PCT:.0%} suspicious rows per class")
axes[1, 1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

,target_4class,rows,p_true_mean,noise_mean,threshold,suspicious_rows,opposite_direction_rate,class_name
0,0,26059,0.387623,0.500019,0.826519,782,0.417629,DOWN_BALANCED
1,1,14842,0.142438,0.500034,0.779399,446,0.565153,DOWN_EXPANSION
2,2,26564,0.363545,0.500019,0.811279,797,0.455654,UP_BALANCED
3,3,11228,0.158963,0.500045,0.780913,337,0.477734,UP_EXPANSION


In [11]:
# Convert suspicious rows into clean/anomaly/review training actions.
score_df["high_conf_opposite"] = (
    score_df["suspicious"]
    & score_df["opposite_direction_prediction"]
    & (score_df["model_confidence"] >= OPPOSITE_HIGH_CONFIDENCE)
)
score_df["same_direction_suspicious"] = score_df["suspicious"] & ~score_df["opposite_direction_prediction"]
score_df["low_conf_opposite_suspicious"] = (
    score_df["suspicious"]
    & score_df["opposite_direction_prediction"]
    & ~score_df["high_conf_opposite"]
)

score_df["training_action"] = "clean"
score_df.loc[score_df["same_direction_suspicious"], "training_action"] = "same_parent_anomaly"
score_df.loc[score_df["low_conf_opposite_suspicious"], "training_action"] = "low_weight_opposite"
score_df.loc[score_df["high_conf_opposite"], "training_action"] = "review_exclude"

score_df["target_8class_anomaly"] = score_df[TARGET_COL].astype(int)
mask_anom = score_df["training_action"] == "same_parent_anomaly"
score_df.loc[mask_anom, "target_8class_anomaly"] = score_df.loc[mask_anom, TARGET_COL].astype(int) + 4
score_df["sample_weight_anomaly"] = 1.0
score_df.loc[score_df["training_action"] == "low_weight_opposite", "sample_weight_anomaly"] = 0.35
score_df.loc[score_df["training_action"] == "review_exclude", "sample_weight_anomaly"] = 0.0

# Attach decisions back to train rows. Rows without OOF scores remain clean.
decision_cols = ["row_id", "noise_score", "training_action", "target_8class_anomaly", "sample_weight_anomaly"]
train8_df = train_df.merge(score_df[decision_cols], on="row_id", how="left")
train8_df["training_action"] = train8_df["training_action"].fillna("clean_no_oof")
train8_df["target_8class_anomaly"] = train8_df["target_8class_anomaly"].fillna(train8_df[TARGET_COL]).astype(int)
train8_df["sample_weight_anomaly"] = train8_df["sample_weight_anomaly"].fillna(1.0).astype(float)
train8_df["target_8class_name"] = train8_df["target_8class_anomaly"].map(CLASS_NAMES_8)

action_summary = train8_df.groupby(["training_action", TARGET_COL]).agg(
    rows=("row_id", "count"),
    avg_weight=("sample_weight_anomaly", "mean"),
).reset_index()
action_summary["class_name"] = action_summary[TARGET_COL].map(CLASS_NAMES_4)
display(action_summary.sort_values(["training_action", TARGET_COL]))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.countplot(data=train8_df, y="training_action", order=train8_df["training_action"].value_counts().index, ax=axes[0])
axes[0].set_title("Training actions")

sns.countplot(data=train8_df, x="target_8class_anomaly", order=list(range(8)), ax=axes[1])
axes[1].set_title("Experimental 8-class training target")
axes[1].set_xticks(range(8))
axes[1].set_xticklabels([str(i) for i in range(8)])

anomaly_over_time = train8_df.assign(is_anomaly=train8_df["target_8class_anomaly"] >= 4).groupby("batch_id")["is_anomaly"].mean().reset_index()
axes[2].plot(anomaly_over_time["batch_id"], anomaly_over_time["is_anomaly"], linewidth=1)
axes[2].set_title("Anomaly share over train batches")
axes[2].set_xlabel("batch_id")
axes[2].set_ylabel("share")
plt.tight_layout()
plt.show()

# Show the most suspicious rows for manual inspection.
inspect_cols = ["timestamp", "batch_id", TARGET_COL, "pred_label", "p_true", "model_confidence", "noise_score", "training_action"]
display(score_df.sort_values("noise_score", ascending=False)[inspect_cols].head(30))

,training_action,target_4class,rows,avg_weight,class_name
0,clean,0,25277,1.00,DOWN_BALANCED
1,clean,1,14396,1.00,DOWN_EXPANSION
2,clean,2,25767,1.00,UP_BALANCED
3,clean,3,10891,1.00,UP_EXPANSION
4,clean_no_oof,0,10657,1.00,DOWN_BALANCED
5,clean_no_oof,1,3827,1.00,DOWN_EXPANSION
6,clean_no_oof,2,9681,1.00,UP_BALANCED
7,clean_no_oof,3,4695,1.00,UP_EXPANSION
8,low_weight_opposite,0,736,0.35,DOWN_BALANCED
9,low_weight_opposite,1,221,0.35,DOWN_EXPANSION


,timestamp,batch_id,target_4class,pred_label,p_true,model_confidence,noise_score,training_action
23033,2025-08-19 00:00:00+00:00,5074,0,2,0.076922,0.658232,0.951119,low_weight_opposite
26598,2025-08-25 19:59:00+00:00,5094,0,2,0.071363,0.827452,0.949524,review_exclude
23034,2025-08-19 00:01:00+00:00,5074,0,2,0.077212,0.640948,0.949436,low_weight_opposite
29508,2025-09-01 00:42:00+00:00,5113,0,2,0.047263,0.799759,0.947304,low_weight_opposite
26597,2025-08-25 19:58:00+00:00,5094,0,2,0.091791,0.779187,0.946531,low_weight_opposite
25890,2025-08-25 00:00:00+00:00,5092,0,2,0.085958,0.741666,0.946262,low_weight_opposite
26596,2025-08-25 19:57:00+00:00,5094,0,2,0.090221,0.786351,0.945755,low_weight_opposite
23035,2025-08-19 00:02:00+00:00,5074,0,2,0.079517,0.647631,0.945724,low_weight_opposite
25932,2025-08-25 00:48:00+00:00,5092,0,2,0.054033,0.827277,0.945585,review_exclude
25891,2025-08-25 00:01:00+00:00,5092,0,2,0.090373,0.741301,0.944768,low_weight_opposite


## Step 7-11: Train 8-Class Anomaly Model And Evaluate By Collapsing Back To 4 Classes

The primary score is still the original 4-class task.

The 8-class model is useful only if, after collapse:

```text
4 -> 0
5 -> 1
6 -> 2
7 -> 3
```

it improves collapsed-4 accuracy, macro F1, direction accuracy, or cross-direction error.

In [12]:
# Train flat 8-class anomaly model on the relabeled training rows.
train8_fit = train8_df[train8_df["sample_weight_anomaly"] > 0].copy()
if train8_fit["target_8class_anomaly"].nunique() < 2:
    raise RuntimeError("The experimental target has fewer than two classes after filtering.")

model8 = train_model(
    train8_fit[selected_features],
    train8_fit["target_8class_anomaly"].astype(int).to_numpy(),
    n_classes=8,
    sample_weight=train8_fit["sample_weight_anomaly"].to_numpy(),
    seed=RANDOM_SEED + 100,
)

comparison_rows = []
prediction_store = {}
for split_name, split_df in [("val", val_df), ("test", test_df)]:
    y_true4 = split_df[TARGET_COL].astype(int).to_numpy()

    base_proba = baseline_predictions[split_name]["proba"]
    base_pred = baseline_predictions[split_name]["pred"]
    base_metrics = compute_eval_metrics(y_true4, base_pred, base_proba)
    comparison_rows.append({"model": "baseline_4class", "split": split_name, **base_metrics})

    proba8 = predict_proba_model(model8, split_df[selected_features])
    pred8 = proba8.argmax(axis=1)
    collapsed_proba4 = collapse_proba_8_to_4(proba8)
    anomaly_metrics = compute_eval_metrics(y_true4, pred8, collapsed_proba4)
    anomaly_metrics["macro_f1_8_proxy_on_collapsed_truth"] = f1_score(
        y_true4,
        pred8,
        labels=list(range(8)),
        average="macro",
        zero_division=0,
    )
    comparison_rows.append({"model": "anomaly_8class_collapsed", "split": split_name, **anomaly_metrics})
    prediction_store[split_name] = {"pred8": pred8, "proba8": proba8, "collapsed_proba4": collapsed_proba4}

comparison_df = pd.DataFrame(comparison_rows)
display(Markdown(f"### 8-class backend: `{model8['backend']}`"))
display(comparison_df)

metric_cols = [c for c in comparison_df.columns if c not in {"model", "split"}]
plot_df = comparison_df.melt(id_vars=["model", "split"], value_vars=metric_cols, var_name="metric", value_name="value")
plt.figure(figsize=(16, 6))
sns.barplot(data=plot_df, x="metric", y="value", hue="model")
plt.xticks(rotation=35, ha="right")
plt.title("Baseline vs anomaly model, primary collapsed-4 evaluation")
plt.tight_layout()
plt.show()

for split_name, split_df in [("val", val_df), ("test", test_df)]:
    plot_confusion(
        split_df[TARGET_COL].astype(int).to_numpy(),
        prediction_store[split_name]["pred8"],
        f"Anomaly 8-class model collapsed to 4 classes: {split_name}",
    )

# Full 8-class prediction distribution is secondary: it tells us whether the model uses anomaly leaves.
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, split_name in zip(axes, ["val", "test"]):
    pred8 = prediction_store[split_name]["pred8"]
    sns.countplot(x=pred8, order=list(range(8)), ax=ax)
    ax.set_title(f"Predicted 8-class leaf distribution: {split_name}")
    ax.set_xlabel("predicted leaf")
plt.tight_layout()
plt.show()

### 8-class backend: `catboost`

,model,split,collapsed_4_accuracy,macro_f1_4,direction_accuracy,cross_direction_error,logloss_4,macro_f1_8_proxy_on_collapsed_truth
0,baseline_4class,val,0.319173,0.242538,0.544712,0.455288,1.404008,NaN
1,anomaly_8class_collapsed,val,0.325979,0.251562,0.563623,0.436377,1.408113,0.125781
2,baseline_4class,test,0.314633,0.218870,0.515344,0.484656,1.420286,NaN
3,anomaly_8class_collapsed,test,0.329471,0.238675,0.534909,0.465091,1.416235,0.119337


In [13]:
# Final decision dashboard for this notebook run.
pivot = comparison_df.pivot(index="split", columns="model")
summary_rows = []
for split_name in ["val", "test"]:
    if split_name not in comparison_df["split"].values:
        continue
    base = comparison_df[(comparison_df["split"] == split_name) & (comparison_df["model"] == "baseline_4class")].iloc[0]
    anom = comparison_df[(comparison_df["split"] == split_name) & (comparison_df["model"] == "anomaly_8class_collapsed")].iloc[0]
    summary_rows.append({
        "split": split_name,
        "accuracy_delta": anom["collapsed_4_accuracy"] - base["collapsed_4_accuracy"],
        "macro_f1_delta": anom["macro_f1_4"] - base["macro_f1_4"],
        "direction_accuracy_delta": anom["direction_accuracy"] - base["direction_accuracy"],
        "cross_direction_error_delta": anom["cross_direction_error"] - base["cross_direction_error"],
        "logloss_delta": anom.get("logloss_4", np.nan) - base.get("logloss_4", np.nan),
    })
decision_df = pd.DataFrame(summary_rows)

def decision_label(row):
    return (
        row["collapsed_4_accuracy_delta"] if "collapsed_4_accuracy_delta" in row else 0
    )

display(Markdown("### Keep/reject decision deltas"))
display(decision_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
if not decision_df.empty:
    decision_plot = decision_df.melt(id_vars="split", var_name="metric", value_name="delta")
    sns.barplot(data=decision_plot, x="metric", y="delta", hue="split", ax=axes[0])
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("Anomaly model minus baseline")
    axes[0].tick_params(axis="x", rotation=35)

criteria = []
for _, row in decision_df.iterrows():
    criteria.append({"split": row["split"], "criterion": "accuracy_up", "pass": row["accuracy_delta"] > 0})
    criteria.append({"split": row["split"], "criterion": "macro_f1_stable_or_up", "pass": row["macro_f1_delta"] >= -0.002})
    criteria.append({"split": row["split"], "criterion": "direction_accuracy_up", "pass": row["direction_accuracy_delta"] > 0})
    criteria.append({"split": row["split"], "criterion": "cross_direction_error_down", "pass": row["cross_direction_error_delta"] < 0})
criteria_df = pd.DataFrame(criteria)
if not criteria_df.empty:
    sns.heatmap(
        criteria_df.pivot(index="criterion", columns="split", values="pass").astype(int),
        annot=True,
        fmt="d",
        cmap="Greens",
        cbar=False,
        ax=axes[1],
    )
    axes[1].set_title("Decision criteria pass map")
plt.tight_layout()
plt.show()

display(Markdown(
    "### Interpretation Reminder\n"
    "Keep this method only if the validation and test deltas improve the collapsed original 4-class objective, "
    "especially cross-direction error. If anomaly leaves are used but collapsed metrics worsen, reject or tighten thresholds."
))

### Keep/reject decision deltas

,split,accuracy_delta,macro_f1_delta,direction_accuracy_delta,cross_direction_error_delta,logloss_delta
0,val,0.006806,0.009024,0.018911,-0.018911,0.004105
1,test,0.014838,0.019805,0.019566,-0.019566,-0.004051


### Interpretation Reminder
Keep this method only if the validation and test deltas improve the collapsed original 4-class objective, especially cross-direction error. If anomaly leaves are used but collapsed metrics worsen, reject or tighten thresholds.